# 00 — API Exploration (نتایج تأییدشده)

**وضعیت:** field name های Euskadi و Open-Meteo کاملاً verify شده‌اند.

## یافته‌های اصلی
| موضوع | نتیجه |
|---|---|
| Euskadi endpoint | `datos_diarios/{STATION}.json` — روزانه، بدون نیاز به token |
| فرمت تاریخ Euskadi | `DD/MM/YYYY` (رشته، نه timestamp) |
| اعداد Euskadi | رشته با کاما به جای نقطه: `"13"` یا `"0,23"` |
| Open-Meteo | تأییدشده: temperature_2m_mean, relative_humidity_2m_mean |
| آخرین رکورد موجود | `14/06/2026` (D-1 موجود است ✅) |


In [1]:
import json
import math
import requests
import pandas as pd
from datetime import date, timedelta
from pathlib import Path

TARGET_DATE = date.today() - timedelta(days=1)   # D-1
YEAR = TARGET_DATE.year
print(f"Testing with date: {TARGET_DATE}  (year={YEAR})")

# ایستگاه‌های پروژه — اسم در parquet = اسم در Euskadi URL
STATIONS = [
    "ALGORTA_BBIZI2",
    "SANTURCE",
    "BASAURI",
    "BARAKALDO",
    "ERANDIO",
    "MAZARREDO",
    "MUSKIZ",
]

Testing with date: 2026-06-14  (year=2026)


---
## Section A — Euskadi: ساختار فایل‌ها

سه نوع فایل per station:
- `datos_horarios/{STATION}.json` — ساعتی
- `datos_diarios/{STATION}.json` — **روزانه ← ما از این استفاده می‌کنیم**
- `datos_indice/{STATION}.json` — شاخص کیفیت هوا

In [2]:
# --- fetch ایستگاه ALGORTA_BBIZI2 و نمایش ساختار واقعی ---
BASE = f"https://opendata.euskadi.eus/contenidos/ds_informes_estudios/calidad_aire_{YEAR}/es_def/adjuntos"

url = f"{BASE}/datos_diarios/ALGORTA_BBIZI2.json"
r = requests.get(url, timeout=30)
print(f"Status: {r.status_code}")

data = r.json()
print(f"\nتعداد رکوردها: {len(data)}")
print(f"\nاولین رکورد (آخرین روز):")
print(json.dumps(data[0], indent=2, ensure_ascii=False))
print(f"\nآخرین رکورد (قدیمی‌ترین):")
print(json.dumps(data[-1], indent=2, ensure_ascii=False))

Status: 200

تعداد رکوردها: 165

اولین رکورد (آخرین روز):
{
  "Date": "14/06/2026",
  "COmgm3": "0,23",
  "CO8hmgm3": "0,23",
  "NOgm3": "2",
  "NOXgm3": "16",
  "NO2gm3": "13",
  "O3gm3": "93",
  "O38hgm3": "87",
  "PM10gm3": "28",
  "PM25gm3": "16",
  "SO2gm3": "4"
}

آخرین رکورد (قدیمی‌ترین):
{
  "Date": "01/01/2026",
  "COmgm3": "0,21",
  "CO8hmgm3": "0,22",
  "NOgm3": "3",
  "NOXgm3": "19",
  "NO2gm3": "14",
  "O3gm3": "42",
  "O38hgm3": "41",
  "PM10gm3": "17",
  "PM25gm3": "7",
  "SO2gm3": "6"
}


In [3]:
# --- تأیید field name ها و parse کردن ---
# فرمت تاریخ: DD/MM/YYYY
# اعداد: رشته (بعضی با کاما: '0,23')

def parse_euskadi_value(val):
    """Convert Euskadi string value to float. Handles '0,23' → 0.23 and missing."""
    if val is None or val == '':
        return float('nan')
    return float(str(val).replace(',', '.'))

def parse_euskadi_date(date_str):
    """DD/MM/YYYY → pd.Timestamp"""
    return pd.to_datetime(date_str, format='%d/%m/%Y')

# تست parse برای اولین رکورد
rec = data[0]
print("Field mapping تأییدشده:")
print(f"  Date    : {rec['Date']} → {parse_euskadi_date(rec['Date']).date()}")
print(f"  NO2gm3  : {rec.get('NO2gm3')}  → NO2  = {parse_euskadi_value(rec.get('NO2gm3'))} µg/m³")
print(f"  PM10gm3 : {rec.get('PM10gm3')} → PM10 = {parse_euskadi_value(rec.get('PM10gm3'))} µg/m³")
print(f"  PM25gm3 : {rec.get('PM25gm3')} → PM2.5= {parse_euskadi_value(rec.get('PM25gm3'))} µg/m³")
print(f"  SO2gm3  : {rec.get('SO2gm3')}  → SO2  = {parse_euskadi_value(rec.get('SO2gm3'))} µg/m³")

Field mapping تأییدشده:
  Date    : 14/06/2026 → 2026-06-14
  NO2gm3  : 13  → NO2  = 13.0 µg/m³
  PM10gm3 : 28 → PM10 = 28.0 µg/m³
  PM25gm3 : 16 → PM2.5= 16.0 µg/m³
  SO2gm3  : 4  → SO2  = 4.0 µg/m³


In [4]:
# --- تست همه 7 ایستگاه برای TARGET_DATE ---
target_str = TARGET_DATE.strftime('%d/%m/%Y')
print(f"بررسی دسترسی D-1 ({target_str}) برای همه ایستگاه‌ها:\n")

results = []
for station in STATIONS:
    url = f"{BASE}/datos_diarios/{station}.json"
    try:
        r = requests.get(url, timeout=20)
        records = r.json()
        # پیدا کردن رکورد D-1
        match = next((rec for rec in records if rec.get('Date') == target_str), None)
        if match:
            no2  = parse_euskadi_value(match.get('NO2gm3'))
            pm10 = parse_euskadi_value(match.get('PM10gm3'))
            pm25 = parse_euskadi_value(match.get('PM25gm3'))
            so2  = parse_euskadi_value(match.get('SO2gm3'))
            results.append({'station': station, 'NO2': no2, 'PM10': pm10, 'PM2.5': pm25, 'SO2': so2, 'status': '✅'})
        else:
            # نمایش آخرین تاریخ موجود
            latest = records[0].get('Date', '?') if records else '?'
            results.append({'station': station, 'NO2': None, 'PM10': None, 'PM2.5': None, 'SO2': None,
                            'status': f'⚠️ آخرین: {latest}'})
    except Exception as e:
        results.append({'station': station, 'NO2': None, 'PM10': None, 'PM2.5': None, 'SO2': None,
                        'status': f'❌ {e}'})

df_check = pd.DataFrame(results)
print(df_check.to_string(index=False))

بررسی دسترسی D-1 (14/06/2026) برای همه ایستگاه‌ها:

       station  NO2  PM10  PM2.5  SO2 status
ALGORTA_BBIZI2 13.0  28.0   16.0  4.0      ✅
      SANTURCE 22.0   NaN   18.0  4.0      ✅
       BASAURI 20.0  26.0   12.0 10.0      ✅
     BARAKALDO 24.0  40.0   17.0  7.0      ✅
       ERANDIO 20.0  25.0   13.0  5.0      ✅
     MAZARREDO 23.0  22.0   13.0  7.0      ✅
        MUSKIZ 10.0  19.0   11.0  8.0      ✅


---
## Section B — Open-Meteo: تأیید field name ها

In [5]:
# test با Algorta/Getxo
url_met = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude":   43.362056,
    "longitude":  -3.022782,
    "start_date": str(TARGET_DATE),
    "end_date":   str(TARGET_DATE),
    "daily": ",".join([
        "temperature_2m_mean",
        "precipitation_sum",
        "windspeed_10m_mean",
        "winddirection_10m_dominant",
        "relative_humidity_2m_mean",
    ]),
    "timezone": "Europe/Madrid",
}
r_met = requests.get(url_met, params=params, timeout=30)
met_raw = r_met.json()
d = met_raw['daily']

print("Open-Meteo fields تأییدشده:")
print(f"  temperature_2m_mean         → Temperature  = {d['temperature_2m_mean'][0]} °C")
print(f"  relative_humidity_2m_mean   → Humidity     = {d['relative_humidity_2m_mean'][0]} %")
print(f"  precipitation_sum           → Precipitation= {d['precipitation_sum'][0]} mm")
print(f"  windspeed_10m_mean          → WindSpeed    = {d['windspeed_10m_mean'][0]} km/h")
print(f"  winddirection_10m_dominant  → WindDirection= {d['winddirection_10m_dominant'][0]} °")

Open-Meteo fields تأییدشده:
  temperature_2m_mean         → Temperature  = 23.9 °C
  relative_humidity_2m_mean   → Humidity     = 60 %
  precipitation_sum           → Precipitation= 0.0 mm
  windspeed_10m_mean          → WindSpeed    = 7.2 km/h
  winddirection_10m_dominant  → WindDirection= 285 °


---
## Section C — تأیید فرمول wind_u / wind_v

In [6]:
def decompose_wind(speed, direction_deg):
    rad = math.radians(direction_deg)
    return -speed * math.sin(rad), -speed * math.cos(rad)

# تست با مقادیر Open-Meteo
speed = d['windspeed_10m_mean'][0]
direction = d['winddirection_10m_dominant'][0]
u, v = decompose_wind(speed, direction)
print(f"WindSpeed={speed}, WindDirection={direction}°")
print(f"  wind_u = {u:.4f}")
print(f"  wind_v = {v:.4f}")

# مقایسه با parquet موجود
parquet_path = Path('../data/processed/air_quality_weather.parquet')
if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    sample = df[df['station']=='ALGORTA_BBIZI2'][['Date','WindSpeed','WindDirection','wind_u','wind_v']].dropna().tail(5)
    sample = sample.copy()
    sample['calc_u'] = sample.apply(lambda r: decompose_wind(r.WindSpeed, r.WindDirection)[0], axis=1)
    sample['calc_v'] = sample.apply(lambda r: decompose_wind(r.WindSpeed, r.WindDirection)[1], axis=1)
    sample['u_ok'] = (sample['wind_u'] - sample['calc_u']).abs() < 0.01
    sample['v_ok'] = (sample['wind_v'] - sample['calc_v']).abs() < 0.01
    print(f"\nمقایسه با parquet (همه باید True باشند):")
    print(sample[['Date','WindSpeed','WindDirection','wind_u','calc_u','u_ok','v_ok']].to_string(index=False))
else:
    print('parquet پیدا نشد — مسیر را تصحیح کن')

WindSpeed=7.2, WindDirection=285°
  wind_u = 6.9547
  wind_v = -1.8635

مقایسه با parquet (همه باید True باشند):
      Date  WindSpeed  WindDirection     wind_u        calc_u  u_ok  v_ok
2026-05-02       16.7            180 -16.700000 -2.045160e-15 False False
2026-05-03       12.7            322  10.007737  7.818901e+00 False False
2026-05-04       16.4            300   8.200000  1.420282e+01 False False
2026-05-05       18.2            283   4.094109  1.773354e+01 False False
2026-05-06       17.7            280   3.073573  1.743110e+01 False False


---
## Section D — نتیجه‌گیری نهایی

**این جدول رو به Claude برگردان:**

In [7]:
print("""
╔══════════════════════════════════════════════════════════╗
║  FIELD MAPPING — تأییدشده از API واقعی                  ║
╠══════════════════════════════════════════════════════════╣
║  Euskadi endpoint pattern:                               ║
║  calidad_aire_{YEAR}/es_def/adjuntos/datos_diarios/      ║
║  {STATION}.json                                          ║
║                                                          ║
║  Euskadi field → parquet column:                         ║
║  'Date'    → Date  (format: DD/MM/YYYY)                  ║
║  'NO2gm3'  → NO2   (string → float, comma decimal)       ║
║  'PM10gm3' → PM10  (string → float)                      ║
║  'PM25gm3' → PM2.5 (string → float)                      ║
║  'SO2gm3'  → SO2   (string → float)                      ║
║                                                          ║
║  Open-Meteo field → parquet column:                      ║
║  'temperature_2m_mean'        → Temperature              ║
║  'relative_humidity_2m_mean'  → Humidity (int)           ║
║  'precipitation_sum'          → Precipitation            ║
║  'windspeed_10m_mean'         → WindSpeed                ║
║  'winddirection_10m_dominant' → WindDirection (int)      ║
║                                                          ║
║  Station name در parquet = Station name در URL           ║
║  (ALGORTA_BBIZI2, SANTURCE, BASAURI, BARAKALDO,          ║
║   ERANDIO, MAZARREDO, MUSKIZ)                            ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║  FIELD MAPPING — تأییدشده از API واقعی                  ║
╠══════════════════════════════════════════════════════════╣
║  Euskadi endpoint pattern:                               ║
║  calidad_aire_{YEAR}/es_def/adjuntos/datos_diarios/      ║
║  {STATION}.json                                          ║
║                                                          ║
║  Euskadi field → parquet column:                         ║
║  'Date'    → Date  (format: DD/MM/YYYY)                  ║
║  'NO2gm3'  → NO2   (string → float, comma decimal)       ║
║  'PM10gm3' → PM10  (string → float)                      ║
║  'PM25gm3' → PM2.5 (string → float)                      ║
║  'SO2gm3'  → SO2   (string → float)                      ║
║                                                          ║
║  Open-Meteo field → parquet column:                      ║
║  'temperature_2m_mean'        → Temperature              ║
║  'relative_humidity_2m